In [6]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from hdbscan import HDBSCAN
from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.en.stop_words import STOP_WORDS

df = pd.read_csv("../0_Data/data_preprocessed_final.csv", sep=",")
df.sample(5)

,index,avis,dates,series,allocine_id,review_id,plateforme,notes,notes_3,nb_capital_words,nb_exclamation_points,nb_exclamation_points_chains,nb_interrogation_points,nb_interrogation_points_chains,lenght,tokens,tokens_lemmatized,sentiment
1356,1358,Better than the reviews would have you believe...,2025-08-30,King & Conqueror,35604,35605569,Imdb,8,1,0,0,0,1,0,399,better reviews would believe good cast interes...,better review would believe good cast interest...,11
255,255,"Great Casting, Small Budget, Too Many Toilet J...",2026-02-04,A Knight of the Seven Kingdoms,28230,35604466,Imdb,7,1,0,0,0,0,0,1469,great casting small budget many toilet jokes s...,great casting small budget many toilet joke sh...,23
533,534,"Fantastic! - After Game of Thrones, I genuinel...",2026-02-18,A Knight of the Seven Kingdoms,28230,35604745,Imdb,10,1,1,2,0,0,0,686,fantastic game thrones genuinely felt depresse...,fantastic game throne genuinely felt depressed...,25
558,559,The core - Return to Westeros. It starts kinda...,2026-02-06,A Knight of the Seven Kingdoms,28230,35604770,Imdb,10,1,1,0,0,0,0,450,core return westeros starts kinda goofy turns ...,core return westeros start kinda goofy turn da...,14
1171,1173,A George R.R. Martin masterpiece - It's just a...,2026-02-23,A Knight of the Seven Kingdoms,28230,35605384,Imdb,10,1,0,0,0,0,0,420,george r r martin masterpiece joy watch pitty ...,george r r martin masterpiece joy watch pitty ...,22


In [8]:
# A faire manuellement pour tous chaque série souhaitée
df = df[df["series"] == "A Knight of the Seven Kingdoms"]

print(df.info(), "\n")

display(df.sample(5))

<class 'pandas.DataFrame'>
RangeIndex: 1237 entries, 0 to 1236
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   index                           1237 non-null   int64
 1   avis                            1237 non-null   str  
 2   dates                           1237 non-null   str  
 3   series                          1237 non-null   str  
 4   allocine_id                     1237 non-null   int64
 5   review_id                       1237 non-null   int64
 6   plateforme                      1237 non-null   str  
 7   notes                           1237 non-null   int64
 8   notes_3                         1237 non-null   int64
 9   nb_capital_words                1237 non-null   int64
 10  nb_exclamation_points           1237 non-null   int64
 11  nb_exclamation_points_chains    1237 non-null   int64
 12  nb_interrogation_points         1237 non-null   int64
 13  nb_interrogati

,index,avis,dates,series,allocine_id,review_id,plateforme,notes,notes_3,nb_capital_words,nb_exclamation_points,nb_exclamation_points_chains,nb_interrogation_points,nb_interrogation_points_chains,lenght,tokens,tokens_lemmatized,sentiment
170,170,Enough - Yes. Game of thrones was and always w...,2026-01-27,A Knight of the Seven Kingdoms,28230,35604381,Imdb,1,-1,8,0,0,3,0,789,enough yes game thrones always masterpiece cer...,enough yes game throne always masterpiece cert...,10
845,847,Peak - Season 1 of this show is the definition...,2026-02-20,A Knight of the Seven Kingdoms,28230,35605058,Imdb,10,1,0,0,0,0,0,658,peak season 1 show definition absolute cinema ...,peak season 1 show definition absolute cinema ...,8
115,115,Wonderful. And beautiful. - A wonderful episod...,2026-01-20,A Knight of the Seven Kingdoms,28230,35604326,Imdb,8,1,0,0,0,0,0,432,wonderful beautiful wonderful episode feeling ...,wonderful beautiful wonderful episode feeling ...,13
1046,1048,Amazing - A Knight of the Seven Kingdoms is pu...,2026-02-21,A Knight of the Seven Kingdoms,28230,35605259,Imdb,10,1,0,0,0,0,0,315,amazing knight seven kingdoms pure character d...,amazing knight seven kingdom pure character dr...,11
423,424,Is this the return of peak television? - After...,2026-02-16,A Knight of the Seven Kingdoms,28230,35604635,Imdb,10,1,1,0,0,1,0,402,return peak television never read book blown a...,return peak television never read book blown a...,6


In [ ]:
# Modélisation

# Isoler les verbatims dans une variable    
verbatims = (
    df["Commentaire"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Modèle de traitement des verbatims
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(verbatims, show_progress_bar=True)

# Doc : https://maartengr.github.io/BERTopic/#installation

In [ ]:

# Fixer les conditions UMAP, notamment pour permettre la reproductibilité
umap_model = UMAP(
    n_neighbors=20,               # Monter à 40 maximum s'il y a trop de clusters
    n_components=5, 
    min_dist=0.0, 
    metric='cosine', 
    random_state=42               # Permet la reproductibilité
)

# Configurer HDBSCAN avec min_cluster_size pour réduire le nombre de topics
hdbscan_model = HDBSCAN(
    min_cluster_size=20,                # Nombre minimum de verbatims par topic
    min_samples=10,                      # Optionnel : contrôle la sensibilité au bruit
    cluster_selection_method="eom",    # cluster plus fins et homogènes
    prediction_data=True
)

# Améliorer la représentation des topics
vectorizer_model = CountVectorizer(stop_words=list(STOP_WORDS), ngram_range=(1, 2), min_df=5)
representation_model = KeyBERTInspired()

# Entrainement du modèle BERTopic
topic_model = BERTopic(embedding_model=embedding_model,
                       umap_model=umap_model, 
                       hdbscan_model=hdbscan_model,
                       vectorizer_model=vectorizer_model,
                       representation_model=representation_model)


topics, probs = topic_model.fit_transform(verbatims, embeddings=embeddings)

# Doc : https://maartengr.github.io/BERTopic/#installation

In [ ]:
# Visualisation des topics
topic_model.visualize_topics()

In [ ]:
# Récupération des mots-clés par topic
topic_info = topic_model.get_topic_info()
print(topic_info)

In [ ]:
# Cliquer ici si besoin de réinitialiser le nombre d'outilers pour itérer
topic_model.update_topics(verbatims, topics=topics)

In [ ]:
# Réduction du nombre d'outliers
print("Nb outlier initialement : ", np.sum(np.array(topic_model.topics_) == -1), "\n")

nouveaux_topics = topic_model.reduce_outliers(
    verbatims,
    topics,
    strategy="embeddings",
    embeddings=embeddings,
    threshold=0.56
)

topic_model.update_topics(verbatims, topics=nouveaux_topics)

nouveaux_topics_info = topic_model.get_topic_info()

print("Nb outliers après : ", np.sum(np.array(topic_model.topics_) == -1), "\n")
print(topic_model.get_topic_info())

In [ ]:
# Visualisation des verbatims par topic
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)

topic_model.visualize_documents(verbatims, reduced_embeddings=reduced_embeddings)

# Le graphique est interactif : il est possible de zoomer pour voir une zone de plus près.

In [ ]:
# Création du dataframe
df_topics = pd.DataFrame({
    "text" : verbatims,
    "topic" : topics,
    "probability" : probs,
    "id_review" : df["id_review"]
})

# Affichage les résultats
print(df_topics)

In [ ]:
# Vue d'ensemble des non classés
df_topics[df_topics["topic"] == -1]

In [ ]:
# Export csv
df_topics.to_csv("../0_Data/topics.csv", sep=";", index=False)